In [1]:
import warnings
warnings.filterwarnings(action='ignore')
import tensorflow.compat.v1 as tf
tf.disable_v2_behavior()
import numpy as np


Instructions for updating:
non-resource variables are not supported in the long term


# 다층 퍼셉트론으로 분류할 MNIST 손글씨 숫자 데이터를 불러와서 전처리한다.

## MNIST 손글씨 실습은 케라스에서 제공하는 데이터셋을 사용한다.

학습 데이터 60,000개 테스트 데이터 10,000가 제공되며 데이터 이미지 하나가 28 * 28인 픽셀 데이터이고 각 픽셀은 흑백으로 0 ~ 255 사이의 값을 가진다.

<img src="./MNIST1.png" width="300" align="left" />

## keras가 제공하는 MNIST 손글씨 숫자 데이터셋을 불러온다.

In [2]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
print(x_train.shape, y_train.shape, x_test.shape, y_test.shape)

(60000, 28, 28) (60000,) (10000, 28, 28) (10000,)


## 학습 데이터를 검증 작업을 하기 위해서 다시 학습 데이터와 검증 데이터로 분리한다.

6만개의 학습 데이터를 학습 데이터(5만개)와 검증 데이터(1만개)로 분리한다.

학습 중간마다 검증 데이터로 모델의 성능을 측정하면 모델 학습이 제대로 진행되는지 검증 정확도를 알 수 있고 학습 정확도는 올라가는데 검증 정확도가 더 이상 올라가지 않거나 오히려 떨어질 경우 조기 종료를 구현할 수 있다.

In [3]:
x_val = x_train[50000:] # 검증 데이터 피처
x_train = x_train[:50000] # 학습 데이터 피처
y_val = y_train[50000:] # 검증 데이터 피처에 따른 레이블
y_train = y_train[:50000] # 학습 데이터 피처에 따른 레이블
print(x_train.shape, x_val.shape, y_train.shape, y_val.shape)

(50000, 28, 28) (10000, 28, 28) (50000,) (10000,)


학습 데이터를 출력해보면 데이터가 0 ~ 255 사이의 값을 가지는 것을 확인할 수 있다.

In [4]:
print(y_train[0])
for row in x_train[0]:
    for i in row:
        print(f'{i:3d} ', end='')
    print()

5
  0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0 
  0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0 
  0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0 
  0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0 
  0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0 
  0   0   0   0   0   0   0   0   0   0   0   0   3  18  18  18 126 136 175  26 166 255 247 127   0   0   0   0 
  0   0   0   0   0   0   0   0  30  36  94 154 170 253 253 253 253 253 225 172 253 242 195  64   0   0   0   0 
  0   0   0   0   0   0   0  49 238 253 253 253 253 253 253 253 253 251  93  82  82  56  39   0   0   0   0   0 
  0   0   0   0   0   0   0  18 219 253 253 253 253 253 198 182 247 241   0   0   0   0   0   

## 다층 퍼셉트론을 사용하는 손글씨 숫자 이미지 분류 작업 흐름도

<img src="./MNIST2.png" width="1200" align="left" />

## 2차원 학습 데이터를 1차원으로 변환한다.

다층 퍼셉트론의 입력값은 무조건 1차원 형태의 배열만 가능하다.  
MNIST 손글씨 숫자 데이터는 2차원이므로 다층 퍼셉트론의 입력값으로 사용할 수 있도록 reshape() 메소드로 1차원 형태로 변환한다.

In [5]:
print(x_train.shape)
# 28행 28열로 구성된 학습 데이터를 784개의 요소를 가지는 1차원 배열로 변환한다.
# np.reshape(x_train, [50000, 784])
# x_train.reshape(50000, 784)
x_train = x_train.reshape(-1, 784) # -1은 행이 개수는 몇개가 나와도 상관없다는 의미이고 784는 열의 개수는 무조건 784개 이다.
print(x_train.shape)
print(x_val.shape)
# 28행 28열로 구성된 검증 데이터를 784개의 요소를 가지는 1차원 배열로 변환한다.
x_val = x_val.reshape(-1, 784)
print(x_val.shape)
print(x_test.shape)
# 28행 28열로 구성된 테스트 데이터를 784개의 요소를 가지는 1차원 배열로 변환한다.
x_test = x_test.reshape(-1, 784)
print(x_test.shape)

(50000, 28, 28)
(50000, 784)
(10000, 28, 28)
(10000, 784)
(10000, 28, 28)
(10000, 784)


## 학습 데이터를 정규화(스케일링)한다.

1차원으로 변경된 데이터를 그대로 다층 퍼셉트론의 입력으로 사용해도 되지만 더 효율적인 학습을 위해서 데이터를 정규화한다. 데이터를 정규화하면 숫자가 작아지므로 모델의 학습 시간을 단축시키고 더 좋은 성능을 보이는 효과가 있다.
MNIST 손글씨 데이터의 모든 값들은 0 ~ 255 범위에 있으므로 255로 나눠서 모든 값들이 0 ~ 1 범위의 값이 되게한다.

MNIST 손글씨 데이터의 각 픽셀의 데이터 타입은 부호 없이 0 ~ 255 범위의 값만 기억하면되므로 데이터 타입이 부호 없는 8비트 정수(uint8)로 되어있다.

In [6]:
print(type(x_train[0][0]))

<class 'numpy.uint8'>


255로 나눠서 0 ~ 1 범위의 실수로 만들어야 하므로 astype() 메소드를 사용해서 실수로 변환한다.

In [7]:
x_train = x_train.astype(np.float32)
print(type(x_train[0][0]))

<class 'numpy.float32'>


In [8]:
x_train /= 255
x_val = x_val.astype(np.float32)
x_val /= 255
x_test = x_test.astype(np.float32)
x_test /= 255

## 레이블 데이터에 One-Hot Encoding을 적용한다.

MNIST 손글씨 숫자 데이터는 0 ~ 9 범위의 숫자로 분류하면 되므로 손실 함수로 크로스 엔트로피를 사용한다.

In [9]:
print(set(y_train))
print(len(set(y_train)))

{np.uint8(0), np.uint8(1), np.uint8(2), np.uint8(3), np.uint8(4), np.uint8(5), np.uint8(6), np.uint8(7), np.uint8(8), np.uint8(9)}
10


크로스 엔트로피를 계산하기 위해서 레이블(y_train, y_val, y_test)을 to_categorical() 메소드로 원-핫 인코딩을 적용한다.  
원-핫 인코딩은 데이터를 수많은 0과 1개의 1로 구별하는 인코딩 방식으로 0으로 이루어진 벡터 집합에서 단 1개의 1의 값으로 해당 데이터를 구별하는 것을 말한다.

to_categorical(원-핫 인코딩을 적용할 데이터, 레이블 개수)

In [10]:
print(y_train[:5])
y_train = tf.keras.utils.to_categorical(y_train, len(set(y_train))) # 학습 데이터에 따른 레이블에 원-핫 인코딩을 적용한다.
for i in y_train[:5]:
    print(i)
y_val = tf.keras.utils.to_categorical(y_val, len(set(y_val))) # 검증 데이터에 따른 레이블에 원-핫 인코딩을 적용한다.
y_test = tf.keras.utils.to_categorical(y_test, len(set(y_test))) # 테스트 데이터에 따른 레이블에 원-핫 인코딩을 적용한다.

[5 0 4 1 9]
[0. 0. 0. 0. 0. 1. 0. 0. 0. 0.]
[1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 1. 0. 0. 0. 0. 0.]
[0. 1. 0. 0. 0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0. 0. 0. 0. 1.]


## 소프트맥스(SoftMax) 작동 원리

argmax() 메소드는 배열에서 가장 큰 값을 찾아서 인덱스를 리턴한다.

In [11]:
a = tf.constant([3, 10, 5]) # 1차원 배열
sess = tf.Session()
print(sess.run(a))
print(sess.run(tf.argmax(a)))

[ 3 10  5]
1


In [12]:
a = tf.constant([[3, 10, 5], [4, 5, 6], [0, 8, 7]]) # 2차원 배열
print(sess.run(a))

[[ 3 10  5]
 [ 4  5  6]
 [ 0  8  7]]


argmax() 메소드를 2차원에서 사용하려면 적절한 2번째 인수를 지정해야 한다. 생략시 기본값은 0이다.  
2번째 인수는 `생략하거나 0`을 지정하면 2차원 배열의 각 `열`에서 최대값의 인덱스를 리턴하고 `1`을 지정하면 2차원 배열의 각 `행`에서 최대값의 인덱스를 리턴한다.

In [13]:
print(sess.run(tf.argmax(a)))
print(sess.run(tf.argmax(a, 0)))

[1 0 2]
[1 0 2]


In [14]:
print(sess.run(tf.argmax(a, 1)))

[1 2 1]


# 전처리된 데이터를 사용하는 다층 퍼셉트론 모델을 만들고 학습시킨다.

## 모델 설명

입력 데이터는 784개의 숫자가 들어있는 1차원 배열 형태이다.  
784개의 입력을 받는 256개의 노드가 1번째 히든 레이어(h1)의 입력으로 사용되고 ReLU 활성화 함수를 거친 1번째 히든 레이어의 출력값을 입력받는 2번째 히든 레이어(h2)에는 128개의 노드가 있다.  
2번째 히든 레이어에는 과적합을 방지하기위해서 10%의 드롭아웃을 적용한다.  
3번째 히든 레이어(logit)는 2번째 히든 레이어의 출력 128를 입력으로 받아 소프트맥스 활성화 함수를 사용하기 위해 10개의 노드가 있다.  
소프트맥스는 분류해야 할 레이블의 총 개수를 k개라고 할 때 k차원의 벡터를 입력받아 각 레이블에 대한 확률을 계산한다.  
예측값(소프트맥스의 출력값)과 실제값의 차이를 계산하기 위해 크로스 엔트로피를 손실 함수로 사용하고 Adam 옵티마이저(최적화 함수)를 사용해서 역전파를 통해 모든 가중치와 바이어스를 최적화 한다.

<img src="./MNIST3.png" width="800" align="left" />

## 다층 퍼셉트론에 사용할 placeholder를 선언한다.

placeholder를 만들 때 2차원 이상의 데이터를 저장해야 할 경우 shape을 이용해서 저장할 데이터의 차원을 지정해야 한다.  
shape의 행 부분을 None으로 지정한 이유는 데이터 개수의 제한없이 입력받기 위해서이다.  
784는 학습 데이터인 28행 28열 이미지를 1차원으로 변경했기 때문이고 10은 레이블에 원-핫 인코딩을 적용했기 때문이다.

In [15]:
X = tf.placeholder(dtype=tf.float32, shape=[None, 784]) # 피처가 저장될 placeholder
Y = tf.placeholder(dtype=tf.float32, shape=[None, 10]) # 레이블이 저장될 placeholder
dropout = tf.placeholder(dtype=tf.float32) # 드롭아웃 비율이 저장될 placeholder

## 다층 퍼셉트론 모델의 히든 레이어 부분을 함수로 만든다.

In [16]:
def mlp(x):
    # 1번째 히든 레이어
    # 784개의 입력을 받는 256개의 뉴런을 사용하기 위해 가중치를 [입력 데이터 수, 뉴런의 개수] 만큼의 난수로 만든다.
    w1 = tf.Variable(tf.random_uniform([784, 256], dtype=tf.float32))
    # 각 뉴런은 1개의 바이어스를 가진다.
    b1 = tf.Variable(tf.zeros([256]))
    # 1번째 히든 레이어의 출력, relu 활성화 함수를 사용한다.
    h1 = tf.nn.relu(tf.matmul(x, w1) + b1)
    
    # 2번째 히든 레이어
    # 1번째 히든 레이어의 출력 256개를 입력받는 128개의 뉴런을 사용하기 위해 가중치를 [입력 데이터 수, 뉴런의 개수] 만큼의 난수로 만든다.
    w2 = tf.Variable(tf.random_uniform([256, 128], dtype=tf.float32))
    # 각 뉴런은 1개의 바이어스를 가진다.
    b2 = tf.Variable(tf.zeros([128]))
    # 2번째 히든 레이어의 출력, relu 활성화 함수를 사용한다.
    h2 = tf.nn.relu(tf.matmul(h1, w2) + b2)
    # 2번째 히든 레이어에는 드롭아웃을 적용시켜 출력한다.
    h2_dropout = tf.nn.dropout(h2, rate=dropout)
    
    # 3번째 히든 레이어
    # 2번째 히든 레이어의 출력 128개를 입력받는 10개의 뉴런을 사용하기 위해 가중치를 [입력 데이터 수, 뉴런의 개수] 만큼의 난수로 만든다.
    w3 = tf.Variable(tf.random_uniform([128, 10], dtype=tf.float32))
    # 각 뉴런은 1개의 바이어스를 가진다.
    b3 = tf.Variable(tf.zeros([10]))
    # 3번째 히든 레이어의 출력, relu 활성화 함수를 사용한다.
    logit = tf.nn.relu(tf.matmul(h2_dropout, w3) + b3)
    
    # 소프트 맥스 활성화 함수에 적용할 값을 리턴시킨다.
    return logit

## 오차 함수와 Adam 옵티마이저를 적용해서 역전파를 구현한다.

softmax_cross_entropy_with_logits_v2() 메소드는 다중 클래스 분류 모델의 손실을 계산할 때 사용되며 소프트맥스 활성화 함수 적용과 크로스 엔트로피 손실 계산을 한 번 처리한다.  
`logits`는 신경망의 마지막 출력층을 통과한 아직 활성화 함수를 거치지 않은 데이터(음수, 양수, 0, 실수)를 넣어준다.  
`labels`는 정답 레이블을 넣어준다.

Adam 옵티마이저는 딥러닝 학습에서 가장 널리 쓰이는 경사 하강법 기반 최적화 알고리즘으로 모멘텀의 이동 방향과 관성을 적응형 학습율을 결합해서 만들어졌다.

In [17]:
# 히든 레이어 부분을 계산하는 함수를 호출한다.
logit = mlp(X)

# 소프트맥스를 적용한 크로스 엔트로피 손실 함수를 만든다.
loss = tf.reduce_mean(
    tf.nn.softmax_cross_entropy_with_logits_v2(logits=logit, labels=Y)
)

# Adam 옵티마이저를 사용해서 모델을 최적화 한다. 모델 최적화 과정은 모델의 예측값과 실제값의 차이를 줄여나가는 과정을 의미한다.
train = tf.train.AdamOptimizer(0.01).minimize(loss)

## 배치 정규화(batch normalization)

배치 정규화는 해당층의 값의 분포를 변경하는 방법으로, 평균과 분산을 고정시키는 방법이다. 배치 정규화 이용하면 기울기 소실을 줄임으로써 신경망 학습 속도를 향상시킬 수 있는 장점이 있다.

## 드롭아웃(dropout)

신경망의 모든 노드를 사용하는 것이 아닌 일부 노드를 사용하는 방법이다.

<img src="./ann11.png" width="1000" align="left" />

왼쪽 그림은 드롭아웃을 적용하기 전 신경망이고, 오른쪽 그림은 드롭아웃을 적용한 신경망이다.

드롭아웃 신경망에서 노드를 `일시적으로 제거`하는 방법이다. 어떤 노드를 신경망에서 제거할지는 각 계층에서 `무작위로 선택`된다. 드롭아웃을 적용하면 신경망의 노드 숫자가 줄어들고 이에 따라 `연산량도 줄어들고 과적합을 방지`할 수 있다는 장점이 있다.

## 학습시킨다.

조기 종료는 과적합을 피하는 충분한 학습을 하기 위해서 학습 중간마다 검증 데이터(x_val)에 대한 정확도를 측정하면 학습 데이터에 대한 정확도는 계속 증가하는 반면에 검증 데이터에 대한 검증 정확도가 점점 떨어질 경우 학습을 중지하는 것을 말한다.

매 epoch 마다 검증 데이터로 검증 정확도를 계산해서 검증 정확도가 5번 연속으로 검증 정확도의 최대값보다 작을 경우 조기 종료를 실행한다.

In [18]:
saver = tf.train.Saver() # tenserflow에서 학습된 모델을 저장하거나 저장된 모델을 불러올 때 사용할 객체를 선언한다.
epoch_cnt = 300 # 조기 종료가 발생되지 않을 경우 최대 학습할 횟수를 지정한다.
batch_size = 1000 # 1번에 읽어서 처리할 학습 데이터 개수를 지정한다. 배치 크기
iteration = len(x_train) // batch_size # batch_size에 따른 1 epoch 당 학습 횟수를 지정한다.
earlystop = 5 # 현재 검증 정확도가 검증 정확도의 최대값보다 5번 연속으로 높지 않을 경우 종료하도록 설정한다.
earlystop_cnt = 0 # 현재 검증 정확도가 검증 정확도의 최대값보다 연속으로 높지 않은 횟수를 세는 변수를 선언한다.

with 구문을 이용해서 세션을 만들면 with 구문 블록의 모든 내용이 실행되고 난 후 자동으로 세션을 종료한다.

In [21]:
with tf.Session() as sess:
    sess.run(tf.global_variables_initializer())
    max_val_acc = 0.0 # 검증 정확도의 최대값을 기억할 변수를 선언한다.
    
    # 지정한 최대 epoch 만큼 반복하며 학습한다.
    # 현재 검증 정확도가 검증 정확도의 최대값보다 5번 연속해서 작게 나올 경우 조기 종료를 실행한다.
    for epoch in range(epoch_cnt):
        avg_loss = 0.0 # epoch 당 손실값을 기억할 변수를 선언한다.
        start = 0 # batch의 시작 인덱스
        end = batch_size # batch의 끝 인덱스
        
        # 1 epoch 학습시 학습 데이터를 batch_size개 만큼씩 나눠서 학습을 진행한다. iteration(batch의 개수) 만큼 반복한다.
        for i in range(iteration):
            _, _loss = sess.run([train, loss], feed_dict={X: x_train[start:end], Y: y_train[start:end], dropout: 0.1})
            # 학습할 batch의 범위를 batch_size개 만큼씩 증가시킨다.
            start += batch_size
            end += batch_size
            # epoch 당 손실값을 계산한다.
            avg_loss += _loss / iteration
        # ===== for i
        
        # 1 epoch의 학습이 종료되면 학습된 모델을 검증 데이터를 이용해서 검증한다.
        # 소프트맥스를 적용해서 예측한다.
        predict = tf.nn.softmax(logit)
        # 정확도를 계산하는 수식을 만든다.
        correct_predict = tf.equal(tf.argmax(Y, 1), tf.argmax(predict, 1))
        accuracy = tf.reduce_mean(tf.cast(correct_predict, dtype=tf.float32))
        
        # Session.run()와 Tensor.eval()의 차이는 Session.run() 세션 외부에서 선언된 내용을 실행하고 Tensor.eval()는 세션 내부에서 선언된 내용을 실행한다.
        # 현재 학습 정확도를 계산한다.
        cur_train_acc = accuracy.eval(feed_dict={X: x_train, Y: y_train, dropout: 0.0})
        # 현재 검증 정확도를 계산한다.
        cur_val_acc = accuracy.eval(feed_dict={X: x_val, Y: y_val, dropout: 0.0})
        # epoch 당 학습 정확도와 검증 정확도를 출력한다.
        print(f'epoch: {epoch + 1:3d}, 현재 학습 정확도: {cur_train_acc:7.5f}, 현재 검증 정확도: {cur_val_acc:7.5f}')
        
        # 최대 검증 정확도와 현재 검증 정확도를 비교해서 조기 종료 조건이 성립하면 모델 학습을 조기 종료시킨다.
        if max_val_acc >= cur_val_acc:
            # 최대 검증 정확도가 현재 검증 정확도 이상일 경우 연속으로 높지 않은 횟수를 세는 변수 earlystop_cnt를 1증가시킨다.
            earlystop_cnt += 1
            print(f'과적합 경고 횟수: {earlystop_cnt}')
            # 현재 검증 정확도가 검증 정확도의 최대값보다 5번 연속으로 높지 않을 경우 종료 조기 종료한다.
            if earlystop_cnt == earlystop:
                print(f'현재 검증 정확도가 최대 검증 정확도 보다 연속으로 {earlystop_cnt}번 이하로 나와서 조기 종료가 실행됨')
                print(f'조기 종료 epoch: {epoch + 1}')
                break
            # ===== if
        else:
            # 최대 검증 정확도가 현재 검증 정확도 미만인 경우
            # 현재 검증 정확도가 최대 검증 정확도 보다 크기 때문에 연속으로 높지 않은 횟수를 세는 변수 earlystop_cnt를 0으로 초기화시킨다.
            earlystop_cnt = 0
            # 최대 검증 정확도를 현재 검증 정확도로 수정한다.
            max_val_acc = cur_val_acc
            # 검증 정확도가 가장 높은 모델을 저장한다.
            saver.save(sess, './model_mlp/model_mlp.ckpt')
        # ===== if

epoch:   1, 현재 학습 정확도: 0.14526, 현재 검증 정확도: 0.14650
epoch:   2, 현재 학습 정확도: 0.57316, 현재 검증 정확도: 0.58810
epoch:   3, 현재 학습 정확도: 0.58644, 현재 검증 정확도: 0.60680
epoch:   4, 현재 학습 정확도: 0.61786, 현재 검증 정확도: 0.63250
epoch:   5, 현재 학습 정확도: 0.65810, 현재 검증 정확도: 0.67660
epoch:   6, 현재 학습 정확도: 0.69168, 현재 검증 정확도: 0.70890
epoch:   7, 현재 학습 정확도: 0.72106, 현재 검증 정확도: 0.73700
epoch:   8, 현재 학습 정확도: 0.74510, 현재 검증 정확도: 0.76090
epoch:   9, 현재 학습 정확도: 0.77034, 현재 검증 정확도: 0.78610
epoch:  10, 현재 학습 정확도: 0.79102, 현재 검증 정확도: 0.80490
epoch:  11, 현재 학습 정확도: 0.80990, 현재 검증 정확도: 0.82400
epoch:  12, 현재 학습 정확도: 0.82606, 현재 검증 정확도: 0.83740
epoch:  13, 현재 학습 정확도: 0.83916, 현재 검증 정확도: 0.84930
epoch:  14, 현재 학습 정확도: 0.84956, 현재 검증 정확도: 0.85630
epoch:  15, 현재 학습 정확도: 0.85908, 현재 검증 정확도: 0.86570
epoch:  16, 현재 학습 정확도: 0.86764, 현재 검증 정확도: 0.87420
epoch:  17, 현재 학습 정확도: 0.87550, 현재 검증 정확도: 0.88070
epoch:  18, 현재 학습 정확도: 0.88176, 현재 검증 정확도: 0.88600
epoch:  19, 현재 학습 정확도: 0.88628, 현재 검증 정확도: 0.89010
epoch:  20, 현재 학습 정확도: 0.89160,

## 검증 결과가 가장 높아서 디스크에 저장한 모델을 불러온다.

In [23]:
with tf.Session() as sess:
    sess.run(tf.global_variables_initializer())
    # 검증 정확도가 가장 높아서 디스크에 저장했던 모델을 불러온다.
    saver.restore(sess, './model_mlp/model_mlp.ckpt')
    predict = tf.nn.softmax(logit)
    correct_predict = tf.equal(tf.argmax(Y, 1), tf.argmax(predict, 1))
    accuracy = tf.reduce_mean(tf.cast(correct_predict, dtype=tf.float32))
    print('학습 정확도: {:7.5f}'.format(accuracy.eval(feed_dict={X: x_train, Y: y_train, dropout: 0.0})))
    print('테스트 정확도: {:7.5f}'.format(accuracy.eval(feed_dict={X: x_test, Y: y_test, dropout: 0.0})))

INFO:tensorflow:Restoring parameters from ./model_mlp/model_mlp.ckpt
학습 정확도: 0.97386
테스트 정확도: 0.93840
